In [123]:
import os
import pandas as pd
from pathlib import Path
from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession

# 1. Move to project root if we are in the notebooks folder
if Path.cwd().name == "notebooks":
    os.chdir("..")

# 2. Initialize Kedro
project_path = Path.cwd()
bootstrap_project(project_path)

# 3. Create session and get catalog
session = KedroSession.create(project_path)
context = session.load_context()
catalog = context.catalog

print(f"✅ Kedro context loaded! Project root: {project_path}")

[04/18/26 21:58:42] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=646148;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=889356;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro_telemetry\plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

✅ Kedro context loaded! Project root: g:\Unidades compartidas\Alianzas\3. Data\CENTRAL\central-perm-flow


# Importación de Fuentes

In [124]:
central_calaca_ext  = catalog.load('central_calendario_extendido')
central_calaca_uptoday  = catalog.load('central_calendario_extendido_uptoday')
central_estaca_sd = catalog.load('central_estaca_sd')

                    INFO     Loading data from central_calendario_extendido                    ]8;id=715816;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=560483;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

                    INFO     Loading data from central_calendario_extendido_uptoday            ]8;id=230279;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=292516;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

                    INFO     Loading data from central_estaca_sd (ParquetDataset)...           ]8;id=188767;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=567115;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [125]:
central_estaca_sd['fecha_ingreso'] = central_estaca_sd['cohorte']

## Bajas

## Parámetros

In [126]:
central_col_fechadef = 'fecha_de_baja_d'
central_col_fechatemp = 'fecha_de_baja_t'

In [127]:
mask_bajas = central_estaca_sd['di'] == 1
central_bajas = central_estaca_sd[mask_bajas]
central_bajas['fecha_baja'] = central_bajas[central_col_fechadef]
central_bajas.loc[central_bajas['fecha_baja'].isna(),'fecha_baja'] = central_bajas.loc[central_bajas['fecha_baja'].isna(),central_col_fechatemp]

# Cruce con el Calendario académico

### Parámetros

In [128]:
# Llaves para el merge temporal
merge_left_on =  'fecha_baja'
merge_right_on =  'fecha_inicio'
# Agrupador para que el tiempo se reinicie por cohorte
group_key = 'fecha_ingreso'
sort_cols = ['fecha_ingreso','fecha_inicio','semana_acumulada']

In [129]:
central_bajas = central_bajas.sort_values(merge_left_on)
central_calaca_uptoday = central_calaca_uptoday.sort_values(merge_right_on)

In [130]:
central_calaca_uptoday.columns


Index(['periodo_raw', 'cohorte', 'cohorte_inicial', 'fecha_ingreso',
       'cohorte_actual', 'bloque', 'fecha_inicio', 'fecha_fin',
       'shifted_fecha_inicio', 'semana', 'semana_acumulada', 'month',
       'mes_academico', 'anio_gregoriano', 'mes_gregoriano', 'student_journey',
       'tipo'],
      dtype='object')

In [131]:

def momento_baja(
    central_estaca_sd: pd.DataFrame,
    central_col_fechadef: str,
    central_col_fechatemp:str,
    central_calaca: pd.DataFrame,
    left_on: str,
    right_on: str,
    group_key: str,
    sort_cols: list
) -> pd.DataFrame:
    """
    Alinea los eventos de deserción con la estructura temporal del calendario académico.

    Utiliza una lógica de 'merge_asof' para situar cada fecha de baja dentro del intervalo 
    académico correcto, permitiendo el análisis de eventos en tiempo relativo (T=0).

    Transformaciones principales:
    1.  **Sincronización Temporal**: Vincula cada 'fecha_baja' con la última 'fecha_inicio' 
        disponible mediante búsqueda hacia atrás (backward search).
    2.  **Reinicio de Reloj por Cohorte**: Asegura que la búsqueda se limite a la 
        'cohorte_inicial' del estudiante, respetando su línea de tiempo específica.
    3.  **Gestión de Bajas Prematuras**: Identifica registros con fechas de baja previas 
        al inicio oficial del calendario, categorizándolos como 'Semana 0'.
    4.  **Enriquecimiento de Etapas**: Hereda las métricas de 'month', 'student_journey' 
        y 'mes_label' correspondientes al momento exacto del retiro.

    Args:
        central_bajas: Dataset de estudiantes que presentan evento de deserción.
        central_calaca: Maestro de calendario académico extendido.
        left_on/right_on: Columnas de fecha para la alineación (baja vs. inicio semana).
        group_key: Llave de agrupación (cohorte) para evitar cruces entre periodos distintos.
        sort_cols: Columnas de ordenamiento para garantizar la integridad del merge temporal.

    Returns:
        pd.DataFrame: Dataset de bajas con su ubicación exacta en la cronología académica.
    """
    # 0. Filtrar bajas
    mask_bajas = central_estaca_sd['di'] == 1
    central_bajas = central_estaca_sd[mask_bajas]
    central_bajas['fecha_baja'] = central_bajas[central_col_fechadef]
    central_bajas.loc[central_bajas['fecha_baja'].isna(),'fecha_baja'] = central_bajas.loc[central_bajas['fecha_baja'].isna(),central_col_fechatemp]
    
    # 1. Preparación: Ordenar es obligatorio para merge_asof
    central_bajas = central_bajas.sort_values(left_on)
    central_calaca = central_calaca.sort_values(right_on)

    # 2. Merge Asof: Busca la última 'fecha_inicio' que sea <= 'fecha_baja'
    # 'by' asegura que el tiempo se reinicie/respete por cohorte
    bajas_calac = pd.merge_asof(
        central_bajas,
        central_calaca,
        left_on=left_on,
        right_on=right_on,
        by=group_key,
        direction="backward" # Busca hacia atrás: la semana que ya inició
    )

    # 3. Manejo de bajas "Pre-Calendario"
    # Si la fecha_baja es menor a la primera fecha_inicio, el merge_asof dejará nulos.
    # Los marcamos como 'Semana 0' o 'Mes 0' (Pre-Onboarding)
    pre_calendar_mask = bajas_calac['semana'].isna()
    
    if pre_calendar_mask.any():
        bajas_calac.loc[pre_calendar_mask, 'semana'] = 0
        bajas_calac.loc[pre_calendar_mask, 'month'] = 0
        bajas_calac.loc[pre_calendar_mask, 'student_journey'] = 'pre-onboarding'
        # Asignamos la cohorte_inicial del estudiante para mantener consistencia
        bajas_calac.loc[pre_calendar_mask, 'mes_label'] = 'm0'

    # 4. Orden final para auditoría
    bajas_calac.sort_values(by= sort_cols, inplace=True)

    return bajas_calac

In [132]:
momento_baja(    central_estaca_sd = central_estaca_sd,
    central_col_fechadef = central_col_fechadef,
    central_col_fechatemp = central_col_fechatemp,
    central_calaca = central_calaca_uptoday,
    left_on = merge_left_on,
    right_on = merge_right_on,
    group_key = group_key,
    sort_cols= sort_cols
                     )

,identificacion,codigo_sis,id_inconcert,nombres,usuario_institucional,alianza,cohorte_x,fecha_ingreso,fecha_de_registro,nivel,...,shifted_fecha_inicio,semana,semana_acumulada,month,mes_academico,anio_gregoriano,mes_gregoriano,student_journey,tipo,mes_label
1,80218031,9.912301e+10,4788.0,henry aldemar zarrate hernandez,hzarrateh@ucentral.edu.co,central,2025-03-17,2025-03-17,2026-04-10,maestria,...,2025-04-07,3.0,3.0,1.0,m1,2025.0,4.0,onboarding,ingreso 1,NaN
2,52097052,9.912301e+10,3734.0,olga lucia nontoa cristancho,onontoac@ucentral.edu.co,central,2025-03-17,2025-03-17,2026-04-10,especializacion,...,2025-04-07,3.0,3.0,1.0,m1,2025.0,4.0,onboarding,ingreso 1,NaN
3,52361727,9.912301e+10,8969.0,sandra paola baquero garcia,sbaquerog1@ucentral.edu.co,central,2025-03-17,2025-03-17,2026-04-10,especializacion,...,2025-04-07,3.0,3.0,1.0,m1,2025.0,4.0,onboarding,ingreso 1,NaN
4,1014255653,9.912301e+10,3389.0,mateo andres rodriguez morales,mrodriguezm@ucentral.edu.co,central,2025-03-17,2025-03-17,2026-04-10,maestria,...,2025-04-07,3.0,3.0,1.0,m1,2025.0,4.0,onboarding,ingreso 1,NaN
5,79671835,9.912301e+10,2562.0,william rene albornoz zoste,walbornozz@ucentral.edu.co,central,2025-03-17,2025-03-17,2026-04-10,maestria,...,2025-04-07,3.0,3.0,1.0,m1,2025.0,4.0,onboarding,ingreso 1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176,53042083,9.912301e+10,128034.0,villamil abaunza julieth,jvillamila@ucentral.edu.co,central,2026-03-16,2026-03-16,2026-04-10,maestria,...,2026-04-13,4.0,4.0,1.0,m1,2026.0,4.0,onboarding,ingreso 1,NaN
177,60361562,9.912301e+10,124500.0,quiroga salazar lola cristina,lquirogas@ucentral.edu.co,central,2026-03-16,2026-03-16,2026-04-10,maestria,...,2026-04-13,4.0,4.0,1.0,m1,2026.0,4.0,onboarding,ingreso 1,NaN
181,52173468,9.912301e+10,127632.0,castellanos lopez sandra liliana,scastellanosl@ucentral.edu.co,central,2026-03-16,2026-03-16,2026-04-10,pregrado,...,2026-04-13,4.0,4.0,1.0,m1,2026.0,4.0,onboarding,ingreso 1,NaN
183,1116276501,9.912301e+10,129811.0,salazar vallejo camilo andres,csalazarv2@ucentral.edu.co,central,2026-03-16,2026-03-16,2026-04-10,pregrado,...,2026-04-13,4.0,4.0,1.0,m1,2026.0,4.0,onboarding,ingreso 1,NaN


In [133]:
bajas_calendario_academico = catalog.load('bajas_calendario_academico')

                    INFO     Loading data from bajas_calendario_academico (ParquetDataset)...  ]8;id=741080;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=997995;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

## Graduados

In [134]:
central_estaca_sd.columns


Index(['identificacion', 'codigo_sis', 'id_inconcert', 'nombres',
       'usuario_institucional', 'alianza', 'cohorte', 'fecha_ingreso',
       'fecha_de_registro', 'nivel', 'nivel_academico', 'programa', 'estado',
       'fecha_de_baja_t', 'fecha_de_baja_d', 'tipo_baja', 'motivo_baja',
       'submotivo_baja', 'comentarios', 'fecha_de_reingreso', 'fecha_grado',
       'exito_estudiantil', 'etapa_studen_journey', 'descuentos', 'di', 'gi'],
      dtype='object')

In [135]:
central_col_fechagrado= 'fecha_grado'
# 0. Filtrar bajas
mask_graduados = central_estaca_sd['gi'] == 1
central_graduados = central_estaca_sd[mask_graduados]


df_merged_go_calendario = pd.merge(
    central_graduados,
    central_calaca_uptoday[central_calaca_uptoday['semana_acumulada']<=32].drop(columns=['cohorte']).groupby(['fecha_ingreso']).agg({'fecha_fin':'max'}),
    on='fecha_ingreso',
    how='left')

df_merged_go_calendario = pd.merge(
    df_merged_go_calendario,
    central_calaca_uptoday.drop(columns=['cohorte']),
    on=['fecha_ingreso','fecha_fin'],
    how='left',
    indicator=True
)



In [137]:
param = catalog.load('parameters')

[04/18/26 21:58:44] INFO     Loading data from parameters (MemoryDataset)...                   ]8;id=394794;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=676510;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [138]:
param.keys()

dict_keys(['data_processing_estaca', 'data_processing_calaca', 'bajas_calac', 'graduados_calac'])

In [168]:
dict_duracion = param['graduados_calac']['dict_niveles_duracion']
dict_duracion


{
    'maestria': {
        'default': {'semanas': 48, 'meses': 12, 'bloques': 6, 'periodos': 3},
        'analitica de datos': {'semanas': 64, 'meses': 16, 'bloques': 8, 'periodos': 4}
    },
    'especializacion': {'default': {'semanas': 32, 'meses': 8, 'bloques': 4, 'periodos': 2}},
    'pregrado': {
        'default': {'semanas': 128, 'meses': 32, 'bloques': 16, 'periodos': 8},
        'derecho': {'semanas': 144, 'meses': 36, 'bloques': 18, 'periodos': 9}
    }
}

In [158]:
dict_duracion


{
    'maestria': {
        'default': {'semanas': 48, 'meses': 12, 'bloques': 6, 'periodos': 3},
        'analitica de datos': {'semanas': 64, 'meses': 16, 'bloques': 8, 'periodos': 4}
    },
    'especializacion': {'default': {'semanas': 32, 'meses': 8, 'bloques': 4, 'periodos': 2}},
    'pregrado': {
        'default': {'semanas': 128, 'meses': 32, 'bloques': 16, 'periodos': 8},
        'derecho': {'semanas': 144, 'meses': 36, 'bloques': 18, 'periodos': 9}
    }
}

In [159]:
config = dict_duracion.get('especializacion', {})
config

{'default': {'semanas': 32, 'meses': 8, 'bloques': 4, 'periodos': 2}}

In [164]:
config = dict_duracion.get('maestria',{})
config


{
    'default': {'semanas': 48, 'meses': 12, 'bloques': 6, 'periodos': 3},
    'analitica de datos': {'semanas': 64, 'meses': 16, 'bloques': 8, 'periodos': 4}
}

In [166]:
programa = 'analitica de datos'
prog_config = config.get(programa, config.get('default', None))
prog_config

{'semanas': 64, 'meses': 16, 'bloques': 8, 'periodos': 4}

In [173]:
def find_graduation_moments(
    central_estaca: pd.DataFrame,
    central_calaca: pd.DataFrame,
    dict_duracion: dict
) -> pd.DataFrame:
    """
    Asigna a los graduados a la última semana teórica de su programa 
    para el análisis de supervivencia.
    """
    # 1. Filtrar solo graduados
    graduados = central_estaca[central_estaca['gi'] == 1].copy()

    # 2. Función interna para obtener la duración según nivel y programa
    def get_duration(row):
        nivel = str(row['nivel']).lower()
        programa = str(row['programa']).lower()
        
        config = dict_duracion.get(nivel, {})
        # Buscar programa específico o usar 'default'
        prog_config = config.get(programa, config.get('default', None))
        
        return prog_config['semanas'] if prog_config else 32 # fallback seguro

    # 3. Asignar la semana máxima teórica a cada estudiante
    graduados['max_semana_teorica'] = graduados.apply(get_duration, axis=1)

    # 4. Merge preciso: Buscamos en el calendario la fila que coincide con 
    # la fecha_ingreso del alumno Y su semana_acumulada teórica.
    df_graduados_cal = pd.merge(
        graduados,
        central_calaca.drop(columns=['cohorte']),
        left_on=['fecha_ingreso', 'max_semana_teorica'],
        right_on=['fecha_ingreso', 'semana_acumulada'],
        how='left'
    )

    return df_graduados_cal

In [174]:
grad_calaca = find_graduation_moments( central_estaca = central_estaca_sd,
                        central_calaca =  central_calaca_uptoday,
                        dict_duracion = dict_duracion)
grad_calaca.loc[:,['nivel','programa','semana_acumulada']].value_counts()


nivel            programa                                  semana_acumulada
especializacion  auditoria y control                       32                  129
maestria         aseguramiento y auditoria de informacion  48                   16
Name: count, dtype: int64